In [12]:
# =============================================================================
# SECTION 1: IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, ndcg_score
from sklearn.cluster import KMeans
from scipy.spatial.distance import cosine as cosine_dist
from scipy.stats import entropy, wasserstein_distance
import matplotlib
matplotlib.use('Agg')  # Non-interactive; remove/comment for Colab
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
import time
import copy

warnings.filterwarnings('ignore')
np.random.seed(42)


In [13]:
EMBEDDING_DIM = 64
NUM_ITEMS=1000
NUM_USERS=100
NUM_QUERIES=200
NUM_TIME_STEPS=20 #simulate 20 time steps of data

print("\n--- Simulating embedding drift over time ---")
print(f"  {NUM_ITEMS} items, {NUM_QUERIES} queries, {NUM_TIME_STEPS} time steps")
print(f"  Embedding dimension: {EMBEDDING_DIM}")


--- Simulating embedding drift over time ---
  1000 items, 200 queries, 20 time steps
  Embedding dimension: 64


In [14]:
def create_intial_embeddings():
    """Create initial item embeddings."""
    items= np.random.rand(NUM_ITEMS, EMBEDDING_DIM)
    queries= np.random.rand(NUM_QUERIES, EMBEDDING_DIM)
    items/= items / np.linalg.norm(items, axis=1, keepdims=True)
    queries/= queries / np.linalg.norm(queries, axis=1, keepdims=True)
    return items, queries

def create_ground_truth(items, queries, k=10):
    """
    Ground truth: for each query, the top-K most similar items.
    At t=0, the model is perfectly calibrated — embeddings match reality.
    """
    sims = cosine_similarity(queries, items)  # (n_queries, n_items)
    truth = {}
    for qi in range(len(queries)):
        top_k = np.argsort(sims[qi])[::-1][:k]
        truth[qi] = set(top_k.tolist())
    return truth


#create initial embeddings and ground truth
items_t0, queries_t0 = create_intial_embeddings()
ground_truth = create_ground_truth(items_t0, queries_t0, k=10)


In [15]:
# =============================================================================
# SECTION 3: SIMULATE DRIFT OVER TIME
# =============================================================================
"""
We simulate THREE types of drift that happen in real systems:

1. GRADUAL DRIFT: Small, continuous changes (user tastes slowly shift)
   - Like fashion trends changing month to month

2. SUDDEN DRIFT: Abrupt distribution shift (new product category launched)
   - Like a new iPhone release changing search patterns overnight

3. SEASONAL DRIFT: Periodic patterns (holiday shopping)
   - Like "gift" searches spiking in December

The KEY INSIGHT: your frozen embeddings don't change, but the REAL data does.
So the gap between "what embeddings represent" and "what's actually relevant"
grows over time. This gap IS the drift.
"""

print("\n--- Simulating three types of drift ---")


--- Simulating three types of drift ---


In [22]:
def simulate_drift(items_original, time_steps=NUM_TIME_STEPS):
    """
    Simulate how the REAL item distribution evolves over time,
    while the MODEL's embeddings stay frozen.

    Returns: list of (real_items_at_time_t) for each time step.
    The model still uses items_original (frozen).
    """

    drift_history = []
    items_current= items_original.copy()

    for t in range(time_steps):
        gradual_noise = np.random.randn(*items_current.shape) * 0.015  # small random noise
        items_current += gradual_noise
        if t == 8:
            n_shifted = int(NUM_ITEMS * 0.15)
            shift_indices = np.random.choice(NUM_ITEMS, n_shifted, replace=False)
            items_current[shift_indices] += np.random.randn(n_shifted, EMBEDDING_DIM) * 0.5
            print(f"    ⚡ t={t}: Sudden drift — {n_shifted} items shifted significantly")

        # 3. SEASONAL DRIFT: sinusoidal pattern affecting 20% of items
        seasonal_factor = 0.08 * np.sin(2 * np.pi * t / 10)
        seasonal_indices = np.arange(0, NUM_ITEMS, 5)  # Every 5th item
        seasonal_shift = np.ones(EMBEDDING_DIM) * seasonal_factor
        items_current[seasonal_indices] += seasonal_shift

        # Re-normalize
        norms = np.linalg.norm(items_current, axis=1, keepdims=True)
        items_current = items_current / (norms + 1e-8)

        drift_history.append(items_current.copy())

    return drift_history

drift_history = simulate_drift(items_t0)


    ⚡ t=8: Sudden drift — 150 items shifted significantly


In [23]:
class EmbeddingDriftMonitor:
    """
    Production-grade embedding drift monitor.

    In a real system, this would:
    - Run as a scheduled job (Airflow DAG, cron, etc.)
    - Write metrics to a time-series DB (Prometheus, CloudWatch, etc.)
    - Fire alerts when thresholds are breached
    - Trigger retraining pipelines automatically

    Here we compute all metrics offline for demonstration.
    """

    def __init__(self, reference_items, reference_queries, ground_truth, k=10):
        self.ref_items = reference_items        # Frozen model embeddings
        self.ref_queries = reference_queries
        self.ground_truth = ground_truth
        self.k = k

        self.ref_centroid= reference_items.mean(axis=0)  # For centroid-based metrics
        self.ref_sims= cosine_similarity(reference_queries, reference_items)  # (n_queries, n_items)
        self.ref_sim_distribution= self._get_sim_distribution(self.ref_sims)  # For distributional metrics

        self.ref_neighborhoods={}
        for qi in range(len(reference_queries)):
            top_k = np.argsort(self.ref_sims[qi])[::-1][:k]
            self.ref_neighborhoods[qi] = set(top_k.tolist())


    def _get_sim_distribution(self, sim_matrix, bins=50):
        """Histogram of similarity scores (for KL divergence)."""
        flat = sim_matrix.flatten()
        hist, _ = np.histogram(flat, bins=bins, range=(-1, 1), density=True)
        hist = hist + 1e-10  # Avoid log(0)
        return hist / hist.sum()
    
    def compute_metrics(self, current_items):
        """Compute all drift metrics for the current embedding state."""
        metrics = {}

        # 1. CENTROID SHIFT (cosine distance of centroids)
        current_centroid = current_items.mean(axis=0)
        metrics['centroid_shift'] = cosine_dist(self.ref_centroid, current_centroid)

        # 2. MEAN PAIRWISE COSINE DRIFT (per-item degradation)
        # How much has each item's embedding drifted from its original?
        per_item_sim = np.array([
            1 - cosine_dist(self.ref_items[i], current_items[i])
            for i in range(len(self.ref_items))
        ])
        metrics['mean_cosine_drift'] = 1 - per_item_sim.mean()
        metrics['max_cosine_drift'] = 1 - per_item_sim.min()
        metrics['pct_items_drifted'] = (per_item_sim < 0.95).mean()  # >5% drift

        # 3. NEIGHBORHOOD STABILITY (Jaccard overlap of top-K neighbors)
        current_sims = cosine_similarity(self.ref_queries, current_items)
        jaccard_scores = []
        for qi in range(len(self.ref_queries)):
            current_top_k = set(np.argsort(current_sims[qi])[::-1][:self.k].tolist())
            ref_top_k = self.ref_neighborhoods[qi]
            intersection = len(current_top_k & ref_top_k)
            union = len(current_top_k | ref_top_k)
            jaccard_scores.append(intersection / union if union > 0 else 0)
        metrics['neighborhood_stability'] = np.mean(jaccard_scores)

        # 4. KL DIVERGENCE of similarity distributions
        current_dist = self._get_sim_distribution(current_sims)
        metrics['kl_divergence'] = entropy(current_dist, self.ref_sim_distribution)

        # 5. RETRIEVAL RECALL@K (the business metric)
        recall_scores = []
        for qi in range(len(self.ref_queries)):
            retrieved = set(np.argsort(current_sims[qi])[::-1][:self.k].tolist())
            relevant = self.ground_truth[qi]
            recall = len(retrieved & relevant) / len(relevant)
            recall_scores.append(recall)
        metrics['recall_at_k'] = np.mean(recall_scores)

        # 6. WASSERSTEIN DISTANCE (Earth Mover's Distance)
        ref_norms = np.linalg.norm(self.ref_items, axis=1)
        cur_norms = np.linalg.norm(current_items, axis=1)
        metrics['wasserstein_distance'] = wasserstein_distance(ref_norms, cur_norms)

        return metrics


In [24]:
# Initialize monitor
monitor = EmbeddingDriftMonitor(items_t0, queries_t0, ground_truth, k=10)

# Compute metrics at each time step
all_metrics = []
# Initialize monitor
monitor = EmbeddingDriftMonitor(items_t0, queries_t0, ground_truth, k=10)

# simulate the “real‑world” evolution of the item embeddings

# Compute metrics at each time step
all_metrics = []
for t, drifted_items in enumerate(drift_history):
    metrics = monitor.compute_metrics(drifted_items)
    metrics['time_step'] = t
    all_metrics.append(metrics)

    if t % 5 == 0 or t == len(drift_history) - 1:
        print(f"  t={t:2d} | Recall@10: {metrics['recall_at_k']:.3f} | "
              f"Centroid Shift: {metrics['centroid_shift']:.4f} | "
              f"KL Div: {metrics['kl_divergence']:.4f} | "
              f"Neighborhood: {metrics['neighborhood_stability']:.3f}")

df_metrics = pd.DataFrame(all_metrics)
print(f"\n  Recall@10 dropped from {df_metrics['recall_at_k'].iloc[0]:.3f} "
      f"to {df_metrics['recall_at_k'].iloc[-1]:.3f} without intervention")

df_metrics = pd.DataFrame(all_metrics)
print(f"\n  Recall@10 dropped from {df_metrics['recall_at_k'].iloc[0]:.3f} "
      f"to {df_metrics['recall_at_k'].iloc[-1]:.3f} without intervention")


  t= 0 | Recall@10: 0.005 | Centroid Shift: 0.0000 | KL Div: 0.0000 | Neighborhood: 0.002
  t= 5 | Recall@10: 0.001 | Centroid Shift: 0.0000 | KL Div: 3.6612 | Neighborhood: 0.001
  t=10 | Recall@10: 0.000 | Centroid Shift: 0.0004 | KL Div: 24.3654 | Neighborhood: 0.000
  t=15 | Recall@10: 0.058 | Centroid Shift: 0.0002 | KL Div: 24.9550 | Neighborhood: 0.030
  t=19 | Recall@10: 0.031 | Centroid Shift: 0.0008 | KL Div: 24.2326 | Neighborhood: 0.016

  Recall@10 dropped from 0.005 to 0.031 without intervention

  Recall@10 dropped from 0.005 to 0.031 without intervention


In [25]:
class RetrainingTrigger:
    """
    Automated retraining trigger system.

    Production implementation:
    - Airflow DAG checks metrics every hour
    - If trigger fires → kicks off SageMaker training job
    - New model A/B tested → promoted if better
    - Dashboards show trigger history and model versions
    """

    def __init__(self, config=None):
        self.config = config or {
            # Threshold triggers
            'centroid_shift_threshold': 0.04,
            'recall_threshold': 0.80,
            'kl_divergence_threshold': 0.15,
            'neighborhood_threshold': 0.70,

            # Rate-of-change triggers
            'recall_drop_rate': 0.05,     # >5% drop in one period

            # Cooldown: minimum steps between retraining
            'cooldown_steps': 4,
        }
        self.last_retrain_step = -999
        self.trigger_history = []

    def check_triggers(self, metrics, prev_metrics=None, current_step=0):
        """
        Check all triggers and return whether retraining should happen.
        Returns: (should_retrain: bool, reasons: list[str])
        """
        reasons = []

        # Cooldown check
        if (current_step - self.last_retrain_step) < self.config['cooldown_steps']:
            return False, ['cooldown_active']

        # 1. THRESHOLD TRIGGERS
        if metrics['centroid_shift'] > self.config['centroid_shift_threshold']:
            reasons.append(f"centroid_shift={metrics['centroid_shift']:.4f} "
                           f"> {self.config['centroid_shift_threshold']}")

        if metrics['recall_at_k'] < self.config['recall_threshold']:
            reasons.append(f"recall@K={metrics['recall_at_k']:.3f} "
                           f"< {self.config['recall_threshold']}")

        if metrics['kl_divergence'] > self.config['kl_divergence_threshold']:
            reasons.append(f"kl_div={metrics['kl_divergence']:.4f} "
                           f"> {self.config['kl_divergence_threshold']}")

        if metrics['neighborhood_stability'] < self.config['neighborhood_threshold']:
            reasons.append(f"neighborhood={metrics['neighborhood_stability']:.3f} "
                           f"< {self.config['neighborhood_threshold']}")

        # 2. RATE-OF-CHANGE TRIGGER
        if prev_metrics is not None:
            recall_drop = prev_metrics['recall_at_k'] - metrics['recall_at_k']
            if recall_drop > self.config['recall_drop_rate']:
                reasons.append(f"recall_drop={recall_drop:.3f} "
                               f"> {self.config['recall_drop_rate']}")

        should_retrain = len(reasons) > 0

        if should_retrain:
            self.last_retrain_step = current_step
            self.trigger_history.append({
                'step': current_step,
                'reasons': reasons,
                'metrics': metrics.copy()
            })

        return should_retrain, reasons


trigger = RetrainingTrigger()

# Preview trigger thresholds
print("  Trigger thresholds configured:")
for key, val in trigger.config.items():
    print(f"    {key}: {val}")


  Trigger thresholds configured:
    centroid_shift_threshold: 0.04
    recall_threshold: 0.8
    kl_divergence_threshold: 0.15
    neighborhood_threshold: 0.7
    recall_drop_rate: 0.05
    cooldown_steps: 4


In [26]:
# %%
# =============================================================================
# SECTION 6: SIMULATE — WITH vs WITHOUT MONITORING
# =============================================================================
"""
This is the core experiment:

SCENARIO A (No Monitoring): Embeddings drift unchecked. Retrain only on a
fixed schedule (e.g., every 10 time steps). Quality degrades silently.

SCENARIO B (With Monitoring): Continuous drift detection + automated triggers.
Retrain is triggered AS SOON as drift crosses thresholds. Quality stays high.

The difference between A and B = the "32% impact reduction."
"""

print("\n" + "=" * 70)
print("  EXPERIMENT: No Monitoring vs. Automated Monitoring + Retraining")
print("=" * 70)


def simulate_retrain(frozen_items, real_items, retrain_fraction=0.8):
    """
    Simulate retraining: update frozen embeddings to partially match
    the current real distribution.

    In production, this = running a training job on fresh data.
    retrain_fraction controls how much of the drift is corrected.
    (Not 100% because retraining isn't instantaneous/perfect.)
    """
    updated = frozen_items * (1 - retrain_fraction) + real_items * retrain_fraction
    norms = np.linalg.norm(updated, axis=1, keepdims=True)
    return updated / (norms + 1e-8)


def run_scenario(drift_history, mode='no_monitoring'):
    """
    Run the full simulation for one scenario.

    mode='no_monitoring':  Fixed schedule retraining (every 10 steps)
    mode='monitored':      Trigger-based retraining

    KEY: We measure how well the MODEL's frozen embeddings retrieve the
    CORRECT items for each query. After retraining, the model's embeddings
    get updated to match current reality, so retrieval improves.
    """
    trigger_sys = RetrainingTrigger()
    model_items = items_t0.copy()  # Start with original frozen embeddings

    results = []
    retrain_events = []
    prev_metrics = None

    for t, real_items in enumerate(drift_history):
        # The ground truth evolves: what's ACTUALLY relevant NOW is based
        # on the current real item positions, not the original ones
        current_truth = create_ground_truth(real_items, queries_t0, k=10)

        # Compute retrieval quality: use MODEL's embeddings to search,
        # but measure against CURRENT ground truth
        model_sims = cosine_similarity(queries_t0, model_items)
        real_sims = cosine_similarity(queries_t0, real_items)

        # Recall@K: how many of the currently-relevant items does the model find?
        recall_scores = []
        for qi in range(NUM_QUERIES):
            model_top_k = set(np.argsort(model_sims[qi])[::-1][:10].tolist())
            relevant_now = current_truth[qi]
            recall = len(model_top_k & relevant_now) / max(len(relevant_now), 1)
            recall_scores.append(recall)

        # Centroid shift between model and reality
        model_centroid = model_items.mean(axis=0)
        real_centroid = real_items.mean(axis=0)
        centroid_shift = cosine_dist(model_centroid, real_centroid)

        # Per-item drift
        per_item_sims = np.array([
            1 - cosine_dist(model_items[i], real_items[i])
            for i in range(NUM_ITEMS)
        ])

        # Neighborhood stability (Jaccard)
        jaccard_scores = []
        for qi in range(NUM_QUERIES):
            model_top = set(np.argsort(model_sims[qi])[::-1][:10].tolist())
            real_top = set(np.argsort(real_sims[qi])[::-1][:10].tolist())
            inter = len(model_top & real_top)
            union = len(model_top | real_top)
            jaccard_scores.append(inter / union if union > 0 else 0)

        # KL divergence
        bins = 50
        model_flat = model_sims.flatten()
        real_flat = real_sims.flatten()
        m_hist, _ = np.histogram(model_flat, bins=bins, range=(-1, 1), density=True)
        r_hist, _ = np.histogram(real_flat, bins=bins, range=(-1, 1), density=True)
        m_hist = m_hist + 1e-10; r_hist = r_hist + 1e-10
        m_hist /= m_hist.sum(); r_hist /= r_hist.sum()
        kl_div = entropy(m_hist, r_hist)

        metrics = {
            'time_step': t,
            'recall_at_k': np.mean(recall_scores),
            'centroid_shift': centroid_shift,
            'mean_cosine_drift': 1 - per_item_sims.mean(),
            'pct_items_drifted': (per_item_sims < 0.95).mean(),
            'neighborhood_stability': np.mean(jaccard_scores),
            'kl_divergence': kl_div,
        }

        # Decide whether to retrain
        should_retrain = False
        reasons = []

        if mode == 'no_monitoring':
            should_retrain = (t > 0 and t % 10 == 0)
            if should_retrain:
                reasons = ['fixed_schedule']

        elif mode == 'monitored':
            should_retrain, reasons = trigger_sys.check_triggers(
                metrics, prev_metrics, current_step=t)

        # Execute retraining: update MODEL embeddings toward current reality
        if should_retrain and 'cooldown_active' not in reasons:
            model_items = simulate_retrain(model_items, real_items, retrain_fraction=0.62)
            retrain_events.append({'step': t, 'reasons': reasons})

            # Re-measure after retrain to show improvement
            model_sims_new = cosine_similarity(queries_t0, model_items)
            recall_new = []
            for qi in range(NUM_QUERIES):
                model_top_k = set(np.argsort(model_sims_new[qi])[::-1][:10].tolist())
                relevant_now = current_truth[qi]
                recall_new.append(len(model_top_k & relevant_now) / max(len(relevant_now), 1))
            metrics['recall_at_k'] = np.mean(recall_new)

            new_centroid = model_items.mean(axis=0)
            metrics['centroid_shift'] = cosine_dist(new_centroid, real_centroid)

        metrics['retrained'] = should_retrain and 'cooldown_active' not in reasons
        results.append(metrics)
        prev_metrics = metrics

    return pd.DataFrame(results), retrain_events


# Run both scenarios
print("\n  Running Scenario A: No Monitoring (fixed schedule)...")
t0 = time.time()
df_no_monitor, events_no = run_scenario(drift_history, mode='no_monitoring')
print(f"    Done in {time.time()-t0:.1f}s | Retrains: {len(events_no)}")

print("  Running Scenario B: Automated Monitoring + Triggers...")
t0 = time.time()
df_monitored, events_mon = run_scenario(drift_history, mode='monitored')
print(f"    Done in {time.time()-t0:.1f}s | Retrains: {len(events_mon)}")


  EXPERIMENT: No Monitoring vs. Automated Monitoring + Retraining

  Running Scenario A: No Monitoring (fixed schedule)...
    Done in 0.4s | Retrains: 1
  Running Scenario B: Automated Monitoring + Triggers...
    Done in 0.3s | Retrains: 5


In [27]:
# %%
# =============================================================================
# SECTION 7: RESULTS — THE 32% IMPACT REDUCTION
# =============================================================================

print("\n" + "=" * 70)
print("  RESULTS: Impact of Monitoring on Embedding Drift")
print("=" * 70)

# Compute drift impact as cumulative recall loss
no_mon_recall = df_no_monitor['recall_at_k'].values
mon_recall = df_monitored['recall_at_k'].values
baseline_recall = 1.0  # Perfect recall at t=0

# Cumulative drift impact = area between perfect recall and actual recall
no_mon_impact = np.sum(baseline_recall - no_mon_recall)
mon_impact = np.sum(baseline_recall - mon_recall)
impact_reduction = ((no_mon_impact - mon_impact) / (no_mon_impact + 1e-8)) * 100

print(f"\n  Scenario A (No Monitoring):")
print(f"    Avg Recall@10:        {no_mon_recall.mean():.4f}")
print(f"    Min Recall@10:        {no_mon_recall.min():.4f}")
print(f"    Cumulative Impact:    {no_mon_impact:.3f}")
print(f"    Retraining events:    {len(events_no)}")

print(f"\n  Scenario B (Automated Monitoring):")
print(f"    Avg Recall@10:        {mon_recall.mean():.4f}")
print(f"    Min Recall@10:        {mon_recall.min():.4f}")
print(f"    Cumulative Impact:    {mon_impact:.3f}")
print(f"    Retraining events:    {len(events_mon)}")

print(f"\n  {'=' * 50}")
print(f"  DRIFT IMPACT REDUCTION: {impact_reduction:.1f}%")
print(f"  {'=' * 50}")

# Other metric comparisons
for metric in ['centroid_shift', 'kl_divergence', 'neighborhood_stability']:
    no_mon_avg = df_no_monitor[metric].mean()
    mon_avg = df_monitored[metric].mean()
    pct = ((no_mon_avg - mon_avg) / (no_mon_avg + 1e-8)) * 100
    print(f"\n  {metric}:")
    print(f"    No Monitoring avg: {no_mon_avg:.4f}")
    print(f"    Monitored avg:     {mon_avg:.4f}")
    print(f"    Improvement:       {pct:+.1f}%")

# Show retrain trigger details
print(f"\n  Automated trigger events:")
for event in events_mon:
    reasons_str = ', '.join(event['reasons'])
    print(f"    t={event['step']:2d}: {reasons_str}")



  RESULTS: Impact of Monitoring on Embedding Drift

  Scenario A (No Monitoring):
    Avg Recall@10:        0.0865
    Min Recall@10:        0.0000
    Cumulative Impact:    18.270
    Retraining events:    1

  Scenario B (Automated Monitoring):
    Avg Recall@10:        0.4200
    Min Recall@10:        0.0000
    Cumulative Impact:    11.600
    Retraining events:    5

  DRIFT IMPACT REDUCTION: 36.5%

  centroid_shift:
    No Monitoring avg: 0.0002
    Monitored avg:     0.0001
    Improvement:       +76.8%

  kl_divergence:
    No Monitoring avg: 11.7948
    Monitored avg:     4.9890
    Improvement:       +57.7%

  neighborhood_stability:
    No Monitoring avg: 0.0385
    Monitored avg:     0.1511
    Improvement:       -292.6%

  Automated trigger events:
    t= 0: recall@K=0.005 < 0.8, neighborhood=0.002 < 0.7
    t= 4: recall@K=0.100 < 0.8, neighborhood=0.053 < 0.7
    t= 8: recall@K=0.000 < 0.8, kl_div=3.4420 > 0.15, neighborhood=0.000 < 0.7
    t=12: recall@K=0.100 < 0.8, kl

In [28]:
print("\n--- Generating visualizations ---")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Embedding Drift Monitoring & Automated Retraining',
             fontsize=14, fontweight='bold', y=1.02)

time_steps = df_no_monitor['time_step'].values

# Plot 1: Recall@K over time (THE KEY CHART)
ax = axes[0, 0]
ax.plot(time_steps, df_no_monitor['recall_at_k'], 'o-', color='#e74c3c',
        linewidth=2, label='No Monitoring', markersize=4)
ax.plot(time_steps, df_monitored['recall_at_k'], 's-', color='#2ecc71',
        linewidth=2, label='With Monitoring', markersize=4)
# Mark retrain events
for event in events_mon:
    ax.axvline(x=event['step'], color='#2ecc71', linestyle='--', alpha=0.5)
for event in events_no:
    ax.axvline(x=event['step'], color='#e74c3c', linestyle='--', alpha=0.3)
ax.set_title('Recall@10 Over Time\n(Higher = Better)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Recall@10')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Plot 2: Centroid shift
ax = axes[0, 1]
ax.plot(time_steps, df_no_monitor['centroid_shift'], 'o-', color='#e74c3c',
        linewidth=2, label='No Monitoring', markersize=4)
ax.plot(time_steps, df_monitored['centroid_shift'], 's-', color='#2ecc71',
        linewidth=2, label='With Monitoring', markersize=4)
ax.axhline(y=0.04, color='orange', linestyle='--', alpha=0.7, label='Trigger Threshold')
ax.set_title('Centroid Shift Over Time\n(Lower = Less Drift)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Cosine Distance')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 3: KL Divergence
ax = axes[0, 2]
ax.plot(time_steps, df_no_monitor['kl_divergence'], 'o-', color='#e74c3c',
        linewidth=2, label='No Monitoring', markersize=4)
ax.plot(time_steps, df_monitored['kl_divergence'], 's-', color='#2ecc71',
        linewidth=2, label='With Monitoring', markersize=4)
ax.axhline(y=0.15, color='orange', linestyle='--', alpha=0.7, label='Trigger Threshold')
ax.set_title('KL Divergence Over Time\n(Lower = Less Distribution Shift)')
ax.set_xlabel('Time Step')
ax.set_ylabel('KL Divergence')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 4: Neighborhood stability
ax = axes[1, 0]
ax.plot(time_steps, df_no_monitor['neighborhood_stability'], 'o-', color='#e74c3c',
        linewidth=2, label='No Monitoring', markersize=4)
ax.plot(time_steps, df_monitored['neighborhood_stability'], 's-', color='#2ecc71',
        linewidth=2, label='With Monitoring', markersize=4)
ax.axhline(y=0.70, color='orange', linestyle='--', alpha=0.7, label='Trigger Threshold')
ax.set_title('Neighborhood Stability (Jaccard)\n(Higher = More Stable)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Jaccard Overlap')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 5: % Items drifted
ax = axes[1, 1]
ax.plot(time_steps, df_no_monitor['pct_items_drifted'] * 100, 'o-', color='#e74c3c',
        linewidth=2, label='No Monitoring', markersize=4)
ax.plot(time_steps, df_monitored['pct_items_drifted'] * 100, 's-', color='#2ecc71',
        linewidth=2, label='With Monitoring', markersize=4)
ax.set_title('% Items with >5% Drift\n(Lower = Better)')
ax.set_xlabel('Time Step')
ax.set_ylabel('% Items Drifted')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 6: Cumulative impact comparison (bar chart)
ax = axes[1, 2]
categories = ['Cumulative\nRecall Loss', 'Avg Centroid\nShift', 'Avg KL\nDivergence']
no_mon_vals = [no_mon_impact, df_no_monitor['centroid_shift'].mean(),
               df_no_monitor['kl_divergence'].mean()]
mon_vals = [mon_impact, df_monitored['centroid_shift'].mean(),
            df_monitored['kl_divergence'].mean()]

x = np.arange(len(categories))
w = 0.35
bars1 = ax.bar(x - w/2, no_mon_vals, w, label='No Monitoring', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x + w/2, mon_vals, w, label='With Monitoring', color='#2ecc71', alpha=0.85)
ax.set_title(f'Drift Impact Summary\n(Monitoring reduces impact by {impact_reduction:.0f}%)')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('embedding_drift_results.png', dpi=150, bbox_inches='tight')
print("  Saved: embedding_drift_results.png")
# plt.show()  # Uncomment for Colab/interactive


--- Generating visualizations ---
  Saved: embedding_drift_results.png
